## 0. Prérequis

1. **Exécution → Modifier le type d'exécution → T4 GPU**.
2. Vérifier que le GPU est bien alloué (cellule suivante).
3. Compter **moins de 10 minutes** par entraînement.

> Toute expérience part de `src/config.py` et ne modifie **qu'un seul**
> hyperparamètre par rapport à la baseline.


In [ ]:
!nvidia-smi

## 1. Cloner le dépôt et installer les dépendances

In [ ]:
import os

REPO_URL = "https://github.com/jaouheraya12-create/transformer-hyperparam-study.git"
REPO_DIR = "transformer-hyperparam-study"

# Idempotent : on clone seulement si le dossier n'existe pas déjà.
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}
!git pull --ff-only


In [ ]:
!pip install -q -r requirements.txt

## 2. Vérifier la configuration

On vérifie que le paquet `src` est importable et que la configuration de
référence est bien celle validée par l'équipe.


In [ ]:
!python -c "import src.config; print('config OK')"

from src.config import Config

cfg = Config()
print(cfg.resume())
print("Splits :", cfg.train_size, "/", cfg.val_size, "/", cfg.test_size)


# 02 — Warmup et scheduler (Fatima)

**Expériences 5 à 8.** Objectif : mesurer l'effet du warmup et de la forme de
décroissance du learning rate.

| Exp. | Nom | `warmup_ratio` | `scheduler_type` |
|---|---|---|---|
| 5 | `warmup_0.1_linear` | 0.1 | linear |
| 6 | `cosine_sans_warmup` | 0.0 | cosine |
| 7 | `warmup_0.1_cosine` | 0.1 | cosine |
| 8 | `warmup_0.2_linear` | 0.2 | linear |

La baseline (exp. 1 : warmup 0.0 + linear) doit être terminée avant d'interpréter.


## Expérience 5 — warmup 0.1 + linear

In [ ]:
!python -m src.train --id 5 --owner Fatima --name warmup_0.1_linear --warmup 0.1


## Expérience 6 — cosine sans warmup

In [ ]:
!python -m src.train --id 6 --owner Fatima --name cosine_sans_warmup --scheduler cosine


## Expérience 7 — warmup 0.1 + cosine

In [ ]:
!python -m src.train --id 7 --owner Fatima --name warmup_0.1_cosine --warmup 0.1 --scheduler cosine


## Expérience 8 — warmup 0.2 + linear

In [ ]:
!python -m src.train --id 8 --owner Fatima --name warmup_0.2_linear --warmup 0.2


## Analyse : comparaison des 4 configurations

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv("results/all_experiments.csv")
mine = df[(df["owner"] == "Fatima") & (df["id"].isin([5, 6, 7, 8]))]

labels = ["warmup 0.1 + linear", "cosine sans warmup",
          "warmup 0.1 + cosine", "warmup 0.2 + linear"]

plt.figure(figsize=(8, 4))
plt.bar(labels, mine["f1"])
plt.ylim(0, 1)
plt.ylabel("F1 (test)")
plt.title("Fatima — warmup et scheduler")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/fatima/figures/f1_warmup_scheduler.png", dpi=150)
plt.show()

mine[["id", "name", "warmup_ratio", "scheduler", "train_time_s", "f1"]]
